# **Phase 3: Feature Engineering**
---
**Scope:** Feature engineering depends on fuel-intrinsic (dry-basis) properties + one plant based performance parameter (moisture).

**Research questions**
1. How to transform raw dry-basis fuel properties into fuel-quality indices?
2. Which engineered indicators would best demonstrate combustion reactivity, energy intensity, and ash-related operational risk?
3. How to incorporate moisture without affecting dry-basis consistency?

## **Engineered feature categories**
---
**1. Fuel-quality indices (energy-related)**

| Feature | Formula | Meaning |
|---|---|---|
| Energy Density Index (EDI) | `CV_db × FC_db` | Energy intensity |
| Volatile-to-Fixed Ratio (VFR) | `VM_db / FC_db` | Reactivity indicator |
| Combustibility Index (CI) | `(FC_db × CV_db) / Ash_db` | Combustion suitability |

**2. Ash behavior indicators (operational risk)**

| Feature | Formula | Relevance |
|---|---|---|
| Alkali Index (AI) | `Na2O + K2O3` | Slagging risk |
| Silica Ratio (SR) | `SiO2 / (CaO + MgO)` | Fouling behavior |
| Base-to-Acid Ratio (B/A) | `(CaO + MgO + Fe2O3) / (SiO2 + Al2O3)` | Ash melting tendency |

**Process-suitability decisions (Phase 4A)**

| Indicator | Interpretation |
|---|---|
| High VM + Low Ash | Pyrolysis |
| High FC + Low Alkali | Combustion |
| Medium VM + Medium Ash | Gasification |


## **Parameters Selection**
---
**Inclusion of primary ash oxides**

* `Na₂O, K₂Ore` the alkali metals with low melting eutectics -> explains slagging and fouling
* `CaO, MgO` are the basic oxides -> increase ash tmperatures
* `SiO₂` react with alkalis -> create sticky silicates
* `Al₂O₃` stablize silicates and affect viscosity
* `Fe₂O₃` is a fluxing agent and have catalytic affects

**Exclusion of secondary ash oxides:** `P2O5`, `TiO2`, `Mn3O4` are not process/system diven parameters in
thermochemical conversion.

In [ ]:
import pandas as pd

import sys
import os
sys.path.append(os.path.abspath(".."))          

from src.features.feature_engineering import (
    add_energy_reactivity_features,
    add_moisture_penalty
)
from src.features.risk_indicators import add_ash_risk_indicators

# Load datasets
df_db = pd.read_csv("../data/interim/harmonized_db_basis.csv")
df_validated = pd.read_csv("../data/interim/validated_data.csv")

# Apply feature engineering
df = add_energy_reactivity_features(df_db)          # df_db is input, store the returned dataframe in df
df = add_ash_risk_indicators(df)                    # ash_risk_indicators are applied on updated df
df = add_moisture_penalty(df, df_validated)         

# Save engineered dataset
df.to_csv("../data/interim/engineered_features.csv", index=False)

df.head()

,Sample_ID,Biomass_Type,Class,Subclass,Ash_db,VM_db,FC_db,C_db,H_db,N_db,...,Latitude,Energy_Density_Index,Volatile_Fixed_Ratio,Combustibility_Index,Alkali_Index,Silica_Ratio,Base_Acid_Ratio,Moist_ar,Moisture_Penalty,Effective_HHV
0,1,Industrial Processing,Timber industry,Woodchips (Softwood),1.1,81.0,17.9,48.6,6.26,0.20,...,-32.193200,367.845,4.525140,334.404545,4.99,0.428990,1.975996,41.2,0.023697,-826.110
1,3,Agricultural,Animal farming,Chicken manure pellets,32.7,56.2,11.1,30.0,3.91,3.94,...,NaN,137.751,5.063063,4.212569,9.03,0.577461,1.623719,18.9,0.050251,-222.139
2,4,Urban Waste,Biosolids,Treated biosolids,42.8,52.0,5.2,24.7,4.35,4.61,...,-33.728392,65.832,10.000000,1.538131,1.72,6.864608,0.717266,8.1,0.109890,-89.886
3,5,Industrial Processing,Paper industry,Paper sludge,26.2,64.2,9.6,32.4,4.96,0.47,...,NaN,135.744,6.687500,5.181069,0.82,1.640852,0.404991,7.8,0.113636,-96.152
4,6,Industrial Processing,Cotton Industry,Cotton seed hulls,1.9,77.9,20.2,32.5,6.02,0.45,...,-30.219856,369.256,3.856436,194.345263,37.97,0.331797,2.703448,11.6,0.079365,-193.768


In [2]:
miss = df.isna().sum().sort_values(ascending=False)
miss

Alkali_Index            39
K2O3                    39
MgO                     38
SiO2                    38
Mn3O4                   38
Other                   38
Base_Acid_Ratio         38
Silica_Ratio            38
Na2O                    38
Al2O3                   38
P2O5                    38
CaO                     38
TiO2                    38
Fe2O3                   38
Longitude               17
Latitude                17
S_db                    16
Effective_HHV           16
Moist_ar                15
Moisture_Penalty        15
Combustibility_Index     6
Energy_Density_Index     6
C_db                     4
VM_db                    4
N_db                     4
CV_MJ/kg_db              4
O_db                     4
FC_db                    4
Volatile_Fixed_Ratio     4
H_db                     4
Biomass_Type             0
Sample_ID                0
Subclass                 0
Class                    0
Ash_db                   0
State                    0
dtype: int64